# Unity Catalog Permissions Model - Complete Guide

## 🎯 Core Concepts to Remember

### Permission Model Characteristics
* **Grant-based only** - No DENY statements (unlike AWS IAM)
* **Additive model** - Users get union of all permissions from all groups
* **Inheritance** - Permissions flow down the hierarchy
* **Principal types** - Users (email) and Groups only (no roles)

---

## 📊 Securable Objects Hierarchy

```
Metastore (root)
├── Catalog
│   ├── Schema (Database)
│   │   ├── Table
│   │   ├── View
│   │   ├── Materialized View
│   │   ├── Function
│   │   └── Volume
│   ├── Connection (External)
│   ├── Storage Credential
│   └── External Location
├── Registered Model
└── Share (Delta Sharing)
```

**Key Point:** Permissions granted at higher levels automatically apply to objects below (unless explicitly revoked).

---

## 🔑 Privileges by Object Type

### 1. **METASTORE** (Top Level)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `CREATE CATALOG` | Create new catalogs | Admins, data engineers |
| `CREATE CONNECTION` | Create external connections | Data engineers |
| `CREATE EXTERNAL LOCATION` | Register external storage | Admins |
| `CREATE STORAGE CREDENTIAL` | Create storage credentials | Admins |
| `USE METASTORE` | Access the metastore | All users |
| `SET SHARE PERMISSION` | Manage Delta Sharing | Admins |

### 2. **CATALOG**
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `USE CATALOG` | See and access catalog | **Required first** - All users |
| `CREATE SCHEMA` | Create schemas in catalog | Data engineers |
| `MODIFY` | Alter catalog properties | Admins |
| `ALL PRIVILEGES` | Full control | Owners |

### 3. **SCHEMA** (Database)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `USE SCHEMA` | See and access schema | **Required first** - All users |
| `CREATE TABLE` | Create tables | Data engineers |
| `CREATE VIEW` | Create views | Data engineers |
| `CREATE MATERIALIZED VIEW` | Create materialized views | Data engineers |
| `CREATE FUNCTION` | Create UDFs | Data engineers |
| `CREATE VOLUME` | Create volumes | Data engineers |
| `MODIFY` | Alter schema properties | Admins |
| `SELECT` | Read from all tables | **Schema-level read access** |
| `ALL PRIVILEGES` | Full control | Owners |

### 4. **TABLE / VIEW**
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `SELECT` | Read data | Analysts, BI tools |
| `MODIFY` | Update/delete data | Data engineers |
| `READ_METADATA` | See table structure only | Metadata tools |
| `ALL PRIVILEGES` | Full control | Owners |

**Important:** `MODIFY` on table includes UPDATE, DELETE, and ALTER operations.

### 5. **VOLUME** (File Storage)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `READ VOLUME` | Read files | Data readers |
| `WRITE VOLUME` | Write/modify files | Data writers |
| `ALL PRIVILEGES` | Full control | Owners |

### 6. **FUNCTION** (UDF)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `EXECUTE` | Run the function | Function users |
| `ALL PRIVILEGES` | Full control | Owners |

### 7. **REGISTERED MODEL** (ML Models)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `EXECUTE` | Use model for inference | ML apps |
| `APPLY TAG` | Add governance tags | Governance teams |
| `ALL PRIVILEGES` | Full control | ML engineers |

### 8. **EXTERNAL LOCATION**
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `CREATE EXTERNAL TABLE` | Create tables on location | Data engineers |
| `CREATE EXTERNAL VOLUME` | Create volumes on location | Data engineers |
| `READ FILES` | Read files from location | Data readers |
| `WRITE FILES` | Write files to location | Data writers |
| `ALL PRIVILEGES` | Full control | Admins |

### 9. **STORAGE CREDENTIAL**
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `CREATE EXTERNAL LOCATION` | Use credential for locations | Admins |
| `ALL PRIVILEGES` | Full control | Security admins |

### 10. **CONNECTION** (External Systems)
| Privilege | Description | Common Use |
|-----------|-------------|------------|
| `CREATE FOREIGN CATALOG` | Create foreign catalog | Data engineers |
| `USE CONNECTION` | Use for external tables | Data engineers |
| `ALL PRIVILEGES` | Full control | Admins |

---

## 🔄 Permission Inheritance Rules

### **The CASCADE Effect**

1. **Catalog-level permissions:**
   - `USE CATALOG` → Required to access anything in the catalog
   - Does NOT automatically grant schema/table access

2. **Schema-level permissions:**
   - `USE SCHEMA` → Required to access objects in schema
   - `SELECT` on schema → SELECT on ALL current tables (not future)
   - `CREATE TABLE` on schema → Can create tables

3. **Object-level permissions:**
   - Most specific, overrides nothing (no DENY model)
   - Must REVOKE to remove

### **Future Grants**
```sql
-- Automatically grant on NEW objects created later
GRANT SELECT ON FUTURE TABLES IN SCHEMA catalog.schema TO `user@example.com`;
```

**Critical for Exam:** FUTURE grants only apply to objects created AFTER the grant is issued.

---

## 💡 Common Permission Patterns

### Pattern 1: Read-Only Access to Schema
```sql
GRANT USAGE ON CATALOG my_catalog TO `analysts`;
GRANT USAGE ON SCHEMA my_catalog.my_schema TO `analysts`;
GRANT SELECT ON SCHEMA my_catalog.my_schema TO `analysts`;
```

### Pattern 2: Data Engineer Access
```sql
GRANT USAGE ON CATALOG my_catalog TO `engineers`;
GRANT USAGE ON SCHEMA my_catalog.my_schema TO `engineers`;
GRANT CREATE TABLE, CREATE VIEW ON SCHEMA my_catalog.my_schema TO `engineers`;
GRANT SELECT, MODIFY ON SCHEMA my_catalog.my_schema TO `engineers`;
```

### Pattern 3: All Tables Except One
```sql
-- Grant on schema
GRANT SELECT ON SCHEMA my_catalog.my_schema TO `team`;
-- Revoke specific table
REVOKE SELECT ON TABLE my_catalog.my_schema.sensitive_table FROM `team`;
```

### Pattern 4: New User Setup
```sql
-- Minimum to see and query data
GRANT USAGE ON CATALOG my_catalog TO `newuser@company.com`;
GRANT USAGE ON SCHEMA my_catalog.my_schema TO `newuser@company.com`;
GRANT SELECT ON TABLE my_catalog.my_schema.my_table TO `newuser@company.com`;
```

---

## 🎓 Certification Quick Reference

### Required Permission Chains

| To Do This | You Need |
|------------|----------|
| **Query a table** | USE CATALOG + USE SCHEMA + SELECT |
| **Create a table** | USE CATALOG + USE SCHEMA + CREATE TABLE |
| **Modify table data** | USE CATALOG + USE SCHEMA + MODIFY |
| **Create a schema** | USE CATALOG + CREATE SCHEMA |
| **Create a catalog** | CREATE CATALOG (metastore) |
| **Use external data** | USE CATALOG + CREATE EXTERNAL TABLE (location) |

### Key Exam Traps

❌ **WRONG:** "DENY blocks access even if granted elsewhere"
✅ **CORRECT:** No DENY exists, must REVOKE from all sources

❌ **WRONG:** "USE CATALOG grants access to all tables"
✅ **CORRECT:** Need USE CATALOG + USE SCHEMA + SELECT separately

❌ **WRONG:** "Future grants apply to existing tables"
✅ **CORRECT:** FUTURE grants only apply to newly created objects

❌ **WRONG:** "Schema SELECT grant affects future tables automatically"
✅ **CORRECT:** Schema-level SELECT only covers existing tables, need FUTURE grant for new ones

---

## 📝 Important SQL Syntax

### Grant Syntax
```sql
GRANT privilege_type ON object_type object_name TO principal;
```

### Revoke Syntax
```sql
REVOKE privilege_type ON object_type object_name FROM principal;
```

### Show Grants
```sql
SHOW GRANTS ON object_type object_name;
SHOW GRANTS TO principal;
```

### Principal Formats
* User: `` `user@example.com` ``
* Group: `` `group_name` ``
* Account users: `` `account users` `` (all workspace users)

---

## 🧠 Memory Tricks for Exam

### The "USE First" Rule
**Remember:** Before you can do ANYTHING with an object, you need USE permission on its parent.

* Query table? → `USE CATALOG` + `USE SCHEMA` + `SELECT`
* Create table? → `USE CATALOG` + `USE SCHEMA` + `CREATE TABLE`
* Alter schema? → `USE CATALOG` + `MODIFY` (on schema)

### The "Privilege Pyramid"
```
        ALL PRIVILEGES
       /              \
    MODIFY          CREATE
      |               |
   SELECT          EXECUTE
      |               |
 READ_METADATA     USE
```

**Bottom to Top:** More restrictive → More permissive

### The "Future vs Present" Rule
* Schema-level `SELECT` = Current tables only
* `FUTURE TABLES` grant = New tables only
* **Want both?** Need BOTH grants!

---

## 📊 Exam Scenario Practice

### Scenario 1: User Can't Query Table
**Problem:** User has SELECT on table but gets "Permission denied"

**Solution Checklist:**
1. ☑️ Do they have `USE CATALOG`?
2. ☑️ Do they have `USE SCHEMA`?
3. ☑️ Do they have `SELECT` on table?
4. ☑️ Is compute allowed to access Unity Catalog?

**Common Exam Answer:** Missing `USE SCHEMA` or `USE CATALOG`

### Scenario 2: Grant Access to All Future Tables
**Question:** Data analyst needs SELECT on all current and future tables in schema.

**Answer:**
```sql
-- Step 1: Access to catalog and schema
GRANT USAGE ON CATALOG prod TO `analyst@company.com`;
GRANT USAGE ON SCHEMA prod.sales TO `analyst@company.com`;

-- Step 2: Existing tables
GRANT SELECT ON SCHEMA prod.sales TO `analyst@company.com`;

-- Step 3: Future tables
GRANT SELECT ON FUTURE TABLES IN SCHEMA prod.sales TO `analyst@company.com`;
```

### Scenario 3: Engineer Creating External Tables
**Question:** What permissions needed to create external table on S3 location?

**Answer:**
```sql
-- 1. Catalog and schema access
GRANT USAGE ON CATALOG my_catalog TO `engineer@company.com`;
GRANT USAGE ON SCHEMA my_catalog.external_data TO `engineer@company.com`;

-- 2. Permission to create tables in schema
GRANT CREATE TABLE ON SCHEMA my_catalog.external_data TO `engineer@company.com`;

-- 3. Permission on external location
GRANT CREATE EXTERNAL TABLE ON EXTERNAL LOCATION `s3_location_name` 
  TO `engineer@company.com`;
```

### Scenario 4: Remove User from Multiple Groups
**Question:** User belongs to groups A, B, C all with SELECT. How to fully revoke?

**Answer:** 
* ❌ Cannot use DENY (doesn't exist)
* ✅ Must remove user from all groups OR revoke from all groups
* ✅ Alternative: Revoke at user level won't work if granted via groups

**Key Point:** Permissions are additive from all group memberships

### Scenario 5: Schema Owner Privileges
**Question:** What can a schema owner do?

**Answer with `ALL PRIVILEGES` on schema:**
* Create/drop tables, views, functions
* Grant permissions to others
* Modify schema properties
* Read all data in schema

**Does NOT automatically include:**
* Create new schemas in catalog (need catalog-level permission)
* Access to other schemas

---

## 🔒 Row-Level and Column-Level Security

### Row Filters (Row-Level Security)
```sql
-- Create row filter function
CREATE FUNCTION catalog.schema.row_filter(region STRING)
RETURN IF(IS_MEMBER('regional_managers'), TRUE, region = current_user());

-- Apply to table
ALTER TABLE catalog.schema.sales
SET ROW FILTER catalog.schema.row_filter ON (region);
```

**Exam Tip:** Row filters use functions that return boolean

### Column Masks (Column-Level Security)
```sql
-- Create masking function
CREATE FUNCTION catalog.schema.ssn_mask(ssn STRING)
RETURN CASE 
  WHEN IS_MEMBER('hr_team') THEN ssn
  ELSE 'XXX-XX-' || SUBSTRING(ssn, 8, 4)
END;

-- Apply to column
ALTER TABLE catalog.schema.employees
ALTER COLUMN ssn SET MASK catalog.schema.ssn_mask;
```

**Exam Tip:** Column masks transform data based on user context

---

## 🎯 Object Ownership

### Owner Privileges
* **Owner** = Creator of object (by default)
* Owner has `ALL PRIVILEGES` automatically
* Owner can transfer ownership:
  ```sql
  ALTER TABLE catalog.schema.table OWNER TO `new_owner@company.com`;
  ```

### Important Ownership Rules
1. Each object has exactly ONE owner (user or group)
2. Owner can do everything, including delete object
3. Owner can grant permissions to others
4. Transferring ownership transfers ALL control

---

## 🚨 Common Mistakes (Exam Traps)

### Mistake 1: Assuming DENY Exists
❌ **Wrong:** "Use DENY to block access even if granted by group"
✅ **Right:** "Remove from group or revoke from all sources"

### Mistake 2: Forgetting USE Chain
❌ **Wrong:** "SELECT privilege is enough to query table"
✅ **Right:** "Need USE CATALOG + USE SCHEMA + SELECT"

### Mistake 3: Future Grants Confusion
❌ **Wrong:** "FUTURE grants apply retroactively"
✅ **Right:** "Only apply to objects created after grant"

### Mistake 4: Schema SELECT Scope
❌ **Wrong:** "Schema SELECT automatically covers future tables"
✅ **Right:** "Schema SELECT only covers existing tables at time of grant"

### Mistake 5: MODIFY vs SELECT
❌ **Wrong:** "MODIFY grants include SELECT automatically"
✅ **Right:** "Need both MODIFY and SELECT for full read-write access"

---

## 📚 Quick Command Reference

### Grant Examples
```sql
-- Catalog access
GRANT USAGE ON CATALOG cat TO `user@example.com`;

-- Schema access and table creation
GRANT USAGE, CREATE TABLE ON SCHEMA cat.sch TO `user@example.com`;

-- Read all tables in schema
GRANT SELECT ON SCHEMA cat.sch TO `user@example.com`;

-- Specific table access
GRANT SELECT ON TABLE cat.sch.table TO `user@example.com`;

-- Future tables
GRANT SELECT ON FUTURE TABLES IN SCHEMA cat.sch TO `user@example.com`;

-- Multiple privileges
GRANT SELECT, MODIFY ON TABLE cat.sch.table TO `user@example.com`;

-- To a group
GRANT SELECT ON TABLE cat.sch.table TO `data_analysts`;

-- All privileges (ownership-like)
GRANT ALL PRIVILEGES ON SCHEMA cat.sch TO `admin@example.com`;
```

### Revoke Examples
```sql
-- Revoke specific privilege
REVOKE SELECT ON TABLE cat.sch.table FROM `user@example.com`;

-- Revoke all from user
REVOKE ALL PRIVILEGES ON SCHEMA cat.sch FROM `user@example.com`;

-- Revoke from group
REVOKE SELECT ON TABLE cat.sch.table FROM `analysts`;
```

### Inspection Commands
```sql
-- See all grants on object
SHOW GRANTS ON TABLE cat.sch.table;
SHOW GRANTS ON SCHEMA cat.sch;
SHOW GRANTS ON CATALOG cat;

-- See grants for a principal
SHOW GRANTS TO `user@example.com`;
SHOW GRANTS TO `group_name`;

-- See ownership
DESCRIBE TABLE EXTENDED cat.sch.table;
-- Look for "Owner" field
```

---

## ✅ Pre-Exam Checklist

Before your certification exam, make sure you can:

- [ ] List all privilege types for tables, schemas, catalogs
- [ ] Explain the USE permission chain requirement
- [ ] Differentiate between schema-level SELECT and FUTURE grants
- [ ] Explain why DENY doesn't exist and implications
- [ ] Write GRANT/REVOKE statements with correct syntax
- [ ] Troubleshoot "permission denied" errors systematically
- [ ] Understand additive permission model from multiple groups
- [ ] Know ownership rules and transfer process
- [ ] Explain row filters vs column masks
- [ ] Remember external location permissions

---

In [0]:
%sql
-- Check what catalogs you can access
SHOW CATALOGS;

-- Check what schemas exist in a catalog (replace 'main' with your catalog)
SHOW SCHEMAS IN main;

-- Check your grants on a specific catalog
SHOW GRANTS ON CATALOG main;

In [0]:
%sql
-- See all tables you can access in a schema
SHOW TABLES IN main.default;

-- Check permissions on a specific table
-- Replace with your actual table name
SHOW GRANTS ON TABLE main.default.your_table_name;

-- Check what grants YOU specifically have (as current user)
SHOW GRANTS TO `{current_user()}`;

In [0]:
%sql
-- EXAMPLE: Setting up a new user with read access
-- (You need admin privileges to run these)

/*
-- Step 1: Grant catalog access
GRANT USAGE ON CATALOG my_catalog TO `newuser@company.com`;

-- Step 2: Grant schema access
GRANT USAGE ON SCHEMA my_catalog.my_schema TO `newuser@company.com`;

-- Step 3: Grant read access to all current tables
GRANT SELECT ON SCHEMA my_catalog.my_schema TO `newuser@company.com`;

-- Step 4: Grant read access to all future tables
GRANT SELECT ON FUTURE TABLES IN SCHEMA my_catalog.my_schema TO `newuser@company.com`;

-- Verify the grants
SHOW GRANTS TO `newuser@company.com`;
*/

In [0]:
%sql
-- EXAMPLE: Setting up a data engineer with full access
-- (You need admin privileges to run these)

/*
-- Catalog access
GRANT USAGE ON CATALOG analytics TO `engineer@company.com`;

-- Schema access and creation rights
GRANT USAGE ON SCHEMA analytics.raw TO `engineer@company.com`;
GRANT CREATE TABLE, CREATE VIEW, CREATE FUNCTION ON SCHEMA analytics.raw 
  TO `engineer@company.com`;

-- Read and write data
GRANT SELECT, MODIFY ON SCHEMA analytics.raw TO `engineer@company.com`;

-- Future grants
GRANT SELECT, MODIFY ON FUTURE TABLES IN SCHEMA analytics.raw 
  TO `engineer@company.com`;

-- Verify
SHOW GRANTS TO `engineer@company.com`;
*/

In [0]:
%sql
-- EXAMPLE: Setting up group-based permissions (BEST PRACTICE)
-- (You need admin privileges to run these)

/*
-- Create or identify groups in Databricks Account Console first
-- Then grant to groups instead of individual users

-- Analysts group - read only
GRANT USAGE ON CATALOG prod TO `analysts`;
GRANT USAGE ON SCHEMA prod.sales TO `analysts`;
GRANT SELECT ON SCHEMA prod.sales TO `analysts`;
GRANT SELECT ON FUTURE TABLES IN SCHEMA prod.sales TO `analysts`;

-- Engineers group - full access
GRANT USAGE ON CATALOG prod TO `engineers`;
GRANT USAGE, CREATE TABLE, CREATE VIEW ON SCHEMA prod.sales TO `engineers`;
GRANT SELECT, MODIFY ON SCHEMA prod.sales TO `engineers`;
GRANT SELECT, MODIFY ON FUTURE TABLES IN SCHEMA prod.sales TO `engineers`;

-- Verify group permissions
SHOW GRANTS TO `analysts`;
SHOW GRANTS TO `engineers`;

-- Users inherit permissions from all groups they belong to
*/

In [0]:
%sql
-- EXAMPLE: Grant access to all tables EXCEPT sensitive ones
-- (You need admin privileges to run these)

/*
-- Step 1: Grant access to entire schema
GRANT USAGE ON CATALOG prod TO `team_lead@company.com`;
GRANT USAGE ON SCHEMA prod.sales TO `team_lead@company.com`;
GRANT SELECT ON SCHEMA prod.sales TO `team_lead@company.com`;

-- Step 2: Revoke access to sensitive table
REVOKE SELECT ON TABLE prod.sales.employee_salaries 
  FROM `team_lead@company.com`;

-- Step 3: Verify they can query other tables
-- SELECT * FROM prod.sales.orders LIMIT 5;  -- This works

-- Step 4: Verify they CANNOT query sensitive table  
-- SELECT * FROM prod.sales.employee_salaries;  -- This fails

-- Check remaining grants
SHOW GRANTS TO `team_lead@company.com`;
*/

## 📝 Self-Test Quiz

Try to answer these before your exam:

### Question 1
**A user has SELECT privilege on a table but gets "Permission Denied" when querying. What are possible causes?**

<details>
<summary>Click for Answer</summary>

**Answer:** Missing one or more of:
1. `USE CATALOG` privilege on the catalog
2. `USE SCHEMA` privilege on the schema
3. Compute cluster doesn't have Unity Catalog access enabled
4. Table has row filter that excludes all rows for this user
</details>

---

### Question 2
**You grant SELECT ON SCHEMA to a user. A new table is created in that schema. Can the user query the new table?**

<details>
<summary>Click for Answer</summary>

**Answer:** NO

Schema-level SELECT only applies to tables that existed at the time of the grant. For new tables, you need:
```sql
GRANT SELECT ON FUTURE TABLES IN SCHEMA catalog.schema TO user;
```
</details>

---

### Question 3
**A user belongs to Group A (has SELECT) and Group B (has no grants). You revoke SELECT from the user directly. Can they still query?**

<details>
<summary>Click for Answer</summary>

**Answer:** YES

The user still has SELECT through Group A membership. Permissions are additive - users get the union of all permissions from all groups. You must:
- Remove them from Group A, OR
- Revoke SELECT from Group A, OR  
- Revoke directly from the user AND remove from groups
</details>

---

### Question 4
**What's the minimum permission needed to create an external table on an external location?**

<details>
<summary>Click for Answer</summary>

**Answer:** You need ALL of:
1. `USE CATALOG` on the catalog
2. `USE SCHEMA` on the schema
3. `CREATE TABLE` on the schema
4. `CREATE EXTERNAL TABLE` on the external location
5. External location must reference a storage credential you have access to
</details>

---

### Question 5
**Can you use DENY to block access granted by another admin?**

<details>
<summary>Click for Answer</summary>

**Answer:** NO

Unity Catalog has NO DENY statement. It uses a grant-only model. To remove access:
- REVOKE the grant
- Remove user from groups that have grants
- Use row filters or column masks for conditional access
</details>

---

### Question 6
**What privileges does a table owner automatically get?**

<details>
<summary>Click for Answer</summary>

**Answer:** `ALL PRIVILEGES` 

This includes:
- SELECT, MODIFY (read/write data)
- GRANT permission to others
- ALTER table structure
- DROP table
- Apply row filters and column masks
- Transfer ownership
</details>

---

### Question 7
**A user needs to read from a table and write results to another table. What minimum privileges are needed?**

<details>
<summary>Click for Answer</summary>

**Answer:**

For source table:
- `USE CATALOG` + `USE SCHEMA` + `SELECT`

For destination table (if exists):
- `USE CATALOG` + `USE SCHEMA` + `MODIFY`

For destination table (if creating new):
- `USE CATALOG` + `USE SCHEMA` + `CREATE TABLE`
</details>

---

### Question 8
**You granted USAGE on catalog and schema, but the user still can't see any tables. Why?**

<details>
<summary>Click for Answer</summary>

**Answer:** 

`USAGE` only grants the ability to ACCESS the container - it doesn't grant any privileges on objects inside.

You also need to grant:
- `SELECT` on schema (for all tables), OR
- `SELECT` on specific tables, OR
- `ALL PRIVILEGES` for full access

`USAGE` alone lets them "see" the catalog/schema exists but not read any data.
</details>

---

# 📝 Unity Catalog Permissions - One-Page Cheat Sheet

## Core Rules
1. **Grant-only model** - No DENY exists
2. **Additive permissions** - Users get union from all groups
3. **USE required first** - Must have USE on parent before accessing children
4. **Inheritance flows down** - But doesn't grant automatically

---

## Essential Privileges by Object

| Object | Key Privileges | Notes |
|--------|----------------|-------|
| **METASTORE** | `CREATE CATALOG`, `USE METASTORE` | Top level |
| **CATALOG** | `USE CATALOG` ⭐, `CREATE SCHEMA` | USE required for all access |
| **SCHEMA** | `USE SCHEMA` ⭐, `CREATE TABLE`, `SELECT`, `MODIFY` | USE required for all access |
| **TABLE** | `SELECT`, `MODIFY`, `READ_METADATA` | MODIFY = UPDATE/DELETE/ALTER |
| **VIEW** | `SELECT` | Same as table |
| **FUNCTION** | `EXECUTE` | To run UDF |
| **VOLUME** | `READ VOLUME`, `WRITE VOLUME` | File storage |
| **EXTERNAL LOCATION** | `CREATE EXTERNAL TABLE`, `READ FILES`, `WRITE FILES` | For external data |

⭐ = Mandatory for any access

---

## Permission Chains (Must Have All)

| Action | Required Privileges |
|--------|--------------------|
| **Query table** | `USE CATALOG` + `USE SCHEMA` + `SELECT` |
| **Insert data** | `USE CATALOG` + `USE SCHEMA` + `MODIFY` |
| **Create table** | `USE CATALOG` + `USE SCHEMA` + `CREATE TABLE` |
| **Create schema** | `USE CATALOG` + `CREATE SCHEMA` |
| **Run function** | `USE CATALOG` + `USE SCHEMA` + `EXECUTE` |
| **External table** | Above + `CREATE EXTERNAL TABLE` on location |

---

## Common Patterns

### Read-Only User
```sql
GRANT USAGE ON CATALOG c TO user;
GRANT USAGE ON SCHEMA c.s TO user;
GRANT SELECT ON SCHEMA c.s TO user;  -- existing tables
GRANT SELECT ON FUTURE TABLES IN SCHEMA c.s TO user;  -- new tables
```

### Data Engineer
```sql
GRANT USAGE ON CATALOG c TO user;
GRANT USAGE, CREATE TABLE, CREATE VIEW ON SCHEMA c.s TO user;
GRANT SELECT, MODIFY ON SCHEMA c.s TO user;
```

### All Except One Table
```sql
GRANT SELECT ON SCHEMA c.s TO user;
REVOKE SELECT ON TABLE c.s.sensitive FROM user;
```

---

## FUTURE Grants

```sql
-- Only applies to NEW objects created AFTER this grant
GRANT SELECT ON FUTURE TABLES IN SCHEMA c.s TO user;
```

**Remember:** 
* Schema-level `SELECT` = existing tables only
* `FUTURE TABLES` = new tables only
* **Want both? Need both grants!**

---

## Syntax Quick Reference

```sql
-- Grant
GRANT privilege ON object_type object_name TO principal;

-- Revoke
REVOKE privilege ON object_type object_name FROM principal;

-- Show grants on object
SHOW GRANTS ON TABLE catalog.schema.table;

-- Show grants to principal
SHOW GRANTS TO `user@example.com`;

-- Transfer ownership
ALTER TABLE catalog.schema.table OWNER TO `new_owner@example.com`;
```

---

## Row & Column Security

### Row Filter
```sql
CREATE FUNCTION filter_func(col STRING) 
RETURN condition;

ALTER TABLE t SET ROW FILTER filter_func ON (col);
```

### Column Mask
```sql
CREATE FUNCTION mask_func(col STRING) 
RETURN masked_value;

ALTER TABLE t ALTER COLUMN col SET MASK mask_func;
```

---

## Exam Traps - Memorize These!

| ❌ Wrong | ✅ Correct |
|----------|------------|
| DENY blocks access | No DENY exists - must REVOKE |
| USE CATALOG grants data access | USE only grants container access |
| Schema SELECT covers future tables | Need separate FUTURE grant |
| MODIFY includes SELECT | Need both for read-write |
| Revoke from user removes all access | User may still have via groups |

---

## Troubleshooting Checklist

**User can't query table:**
- [ ] `USE CATALOG`?
- [ ] `USE SCHEMA`?
- [ ] `SELECT` on table or schema?
- [ ] Compute has Unity Catalog access?
- [ ] Row filter allowing rows?

**Can't create table:**
- [ ] `USE CATALOG`?
- [ ] `USE SCHEMA`?
- [ ] `CREATE TABLE` on schema?
- [ ] For external: permission on external location?

**Can't see new tables:**
- [ ] Need `SELECT ON FUTURE TABLES`
- [ ] Schema-level SELECT only covered existing tables

---

## Ownership
* Each object has ONE owner (user or group)
* Owner gets `ALL PRIVILEGES` automatically
* Owner can transfer ownership
* Creator = default owner

---

## Principal Formats
* **User:** `` `user@example.com` ``
* **Group:** `` `group_name` ``
* **All users:** `` `account users` ``

---

## Remember: The 3-Step Pattern
1. 🔑 **USE** - Access to containers (catalog, schema)
2. 🛠️ **ACTION** - What to do (SELECT, CREATE, MODIFY)
3. 📅 **FUTURE** - Don't forget future objects if needed!